In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / 'common' / 'scripts'))
from metabase_client import MetabaseClient

# ── Parameters ──
NAV_DATE = '2026-03-03'
DEPOSITORY_FEE_RATE = 0.00025  # 0.025% p.a.

FUNDS = [
    {'code': 'TUV100', 'name': 'Tuleva III Samba Pensionifond', 'isin': 'EE3600001707'},
    {'code': 'TKF100', 'name': 'Tuleva Täiendav Kogumisfond', 'isin': 'EE0000003283'},
]

# One-off adjustments (outstanding invoices etc.)
ADJUSTMENTS = {
    'TUV100': -9676.30,  # unpaid February depositary fee invoice
}

client = MetabaseClient()
print(f'NAV date: {NAV_DATE}')
for f in FUNDS:
    adj = ADJUSTMENTS.get(f['code'], 0)
    adj_str = f'  (adj: {adj:+,.2f} EUR)' if adj else ''
    print(f'  {f["code"]}: {f["name"]}{adj_str}')

NAV date: 2026-03-03
  TUV100: Tuleva III Samba Pensionifond  (adj: -9,676.30 EUR)
  TKF100: Tuleva Täiendav Kogumisfond


In [2]:
# ── Fetch all data from Metabase ──
raw_positions = pd.DataFrame(client.execute_card(2296))
raw_index = pd.DataFrame(client.execute_card(2297))
raw_units = pd.DataFrame(client.execute_card(2298))
raw_registry = pd.DataFrame(client.execute_card(2299))

print(f'Card 2296 (positions):  {len(raw_positions)} rows')
print(f'Card 2297 (index):      {len(raw_index)} rows')
print(f'Card 2298 (units):      {len(raw_units)} rows')
print(f'Card 2299 (registry):   {len(raw_registry)} rows')

# ── Prepare per-fund data ──
fund_data = {}
for fund in FUNDS:
    fd = {}
    code = fund['code']

    all_positions = raw_positions[
        (raw_positions['Fund Code'] == code) &
        (raw_positions['Nav Date'] == NAV_DATE)
    ].copy()

    fd['all_positions'] = all_positions
    fd['securities'] = all_positions[all_positions['Account Type'] == 'SECURITY'].copy()
    fd['cash'] = all_positions[all_positions['Account Type'] == 'CASH'].copy()
    fd['receivables'] = all_positions[all_positions['Account Type'] == 'RECEIVABLES'].copy()
    fd['liabilities'] = all_positions[all_positions['Account Type'] == 'LIABILITY'].copy()
    fd['units_row'] = all_positions[all_positions['Account Type'] == 'UNITS'].copy()

    print(f'\n{code} on {NAV_DATE}: {len(all_positions)} rows  '
          f'({len(fd["securities"])} sec, {len(fd["cash"])} cash, '
          f'{len(fd["receivables"])} recv, {len(fd["liabilities"])} liab, {len(fd["units_row"])} units)')

    # Units data — try card 2298 first, fall back to index data (card 2297)
    all_fund_units = raw_units[raw_units['Security Name'] == fund['name']].copy()
    all_fund_units = all_fund_units.sort_values('Request Date', ascending=False)
    fd['units_today'] = all_fund_units[all_fund_units['Request Date'] == NAV_DATE]

    # Index data for this fund's ISIN (used as fallback for reported NAV)
    isin_key = fund['isin']
    fund_idx = raw_index[raw_index['Key'] == isin_key].copy()
    fund_idx['Date'] = pd.to_datetime(fund_idx['Date'])
    fund_idx = fund_idx.sort_values('Date', ascending=False)
    fd['index_nav_series'] = fund_idx

    # Determine reported NAV for today
    if len(fd['units_today']) > 0:
        fd['reported_nav'] = fd['units_today'].iloc[0]['Nav']
        fd['nav_source'] = 'units card'
    else:
        # Fall back to index data — latest available value
        nav_dt_ts = pd.to_datetime(NAV_DATE)
        idx_on_date = fund_idx[fund_idx['Date'] <= nav_dt_ts]
        if len(idx_on_date) > 0:
            fd['reported_nav'] = idx_on_date.iloc[0]['Value']
            fd['reported_nav_date'] = idx_on_date.iloc[0]['Date'].strftime('%Y-%m-%d')
            fd['nav_source'] = f'index data ({fd["reported_nav_date"]})'
        else:
            fd['reported_nav'] = None
            fd['nav_source'] = 'unavailable'

    # Previous NAV for day-over-day comparison
    prev_dates = all_fund_units[all_fund_units['Request Date'] < NAV_DATE]['Request Date'].unique()
    prev_date = None
    if len(prev_dates) > 0 and fd['reported_nav'] is not None:
        today_nav_val = fd['reported_nav']
        for d in sorted(prev_dates, reverse=True):
            row = all_fund_units[all_fund_units['Request Date'] == d].iloc[0]
            if row['Nav'] != today_nav_val:
                prev_date = d
                break
        if prev_date is None:
            prev_date = sorted(prev_dates)[-1]

    # If no units card history, use index data for prev NAV
    if prev_date is None and len(fund_idx) >= 2:
        nav_dt_ts = pd.to_datetime(NAV_DATE)
        older = fund_idx[fund_idx['Date'] < nav_dt_ts]
        if len(older) > 0:
            prev_date = older.iloc[0]['Date'].strftime('%Y-%m-%d')
            fd['prev_nav_from_index'] = older.iloc[0]['Value']
            if len(older) >= 2:
                fd['prev_prev_nav_from_index'] = older.iloc[1]['Value']

    fd['prev_date'] = prev_date
    fd['units_prev'] = all_fund_units[all_fund_units['Request Date'] == prev_date] if (prev_date and len(all_fund_units) > 0) else pd.DataFrame()
    print(f'  Units dates: today={NAV_DATE}, prev={prev_date}')
    print(f'  Reported NAV source: {fd["nav_source"]}')

    fund_reg = raw_registry[raw_registry['Isin'] == fund['isin']]
    if len(fund_reg) > 0:
        fd['mgmt_fee_rate'] = fund_reg.iloc[0]['Management Fee Rate']
        print(f'  Mgmt fee: {fd["mgmt_fee_rate"]*100:.3f}% p.a.')
    else:
        fd['mgmt_fee_rate'] = None
        print(f'  WARNING: Fund not found in registry!')

    print(f'  Depository fee: {DEPOSITORY_FEE_RATE*100:.3f}% p.a.')

    fund_data[code] = fd

Card 2296 (positions):  245 rows
Card 2297 (index):      537 rows
Card 2298 (units):      18 rows
Card 2299 (registry):   58 rows

TUV100 on 2026-03-03: 12 rows  (6 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units dates: today=2026-03-03, prev=2026-03-02
  Reported NAV source: units card
  Mgmt fee: 0.179% p.a.
  Depository fee: 0.025% p.a.

TKF100 on 2026-03-03: 15 rows  (9 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units dates: today=2026-03-03, prev=2026-03-02
  Reported NAV source: index data (2026-03-03)
  Mgmt fee: 0.150% p.a.
  Depository fee: 0.025% p.a.


## Fund Position

All holdings of the fund on the NAV date, split into securities (ETFs) and cash/accrual lines.

In [3]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']
    cash = fd['cash']
    receivables = fd['receivables']
    liabilities = fd['liabilities']
    units_row = fd['units_row']

    sec_display = securities[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value']].copy()
    sec_display = sec_display.sort_values('Market Value', ascending=False)
    sec_total = sec_display['Market Value'].sum()

    cash_total = cash['Market Value'].sum()
    recv_total = receivables['Market Value'].sum()
    liab_total = liabilities['Market Value'].sum()
    other_rows = [
        {'Account ID': '', 'Account Name': 'Cash', 'Quantity': None, 'Market Price': None, 'Market Value': cash_total},
        {'Account ID': '', 'Account Name': 'Receivables', 'Quantity': None, 'Market Price': None, 'Market Value': recv_total},
        {'Account ID': '', 'Account Name': 'Liabilities', 'Quantity': None, 'Market Price': None, 'Market Value': liab_total},
    ]

    all_display = pd.concat([sec_display, pd.DataFrame(other_rows)], ignore_index=True)

    # Previous working day comparison
    available_dates = sorted(raw_positions[raw_positions['Fund Code'] == code]['Nav Date'].unique())
    nav_idx = available_dates.index(NAV_DATE) if NAV_DATE in available_dates else -1
    prev_nav_date = available_dates[nav_idx - 1] if nav_idx > 0 else None

    if prev_nav_date:
        prev_pos = raw_positions[
            (raw_positions['Fund Code'] == code) &
            (raw_positions['Nav Date'] == prev_nav_date)
        ]
        prev_sec = prev_pos[prev_pos['Account Type'] == 'SECURITY'].set_index('Account ID')['Market Value']
        prev_cash = prev_pos[prev_pos['Account Type'] == 'CASH']['Market Value'].sum()
        prev_recv = prev_pos[prev_pos['Account Type'] == 'RECEIVABLES']['Market Value'].sum()
        prev_liab = prev_pos[prev_pos['Account Type'] == 'LIABILITY']['Market Value'].sum()
        prev_by_type = {'Cash': prev_cash, 'Receivables': prev_recv, 'Liabilities': prev_liab}

        prev_values = []
        for _, row in all_display.iterrows():
            if row['Account ID'] and row['Account ID'] in prev_sec.index:
                prev_values.append(prev_sec[row['Account ID']])
            elif row['Account Name'] in prev_by_type:
                prev_values.append(prev_by_type[row['Account Name']])
            else:
                prev_values.append(None)

        all_display['Prev Value'] = prev_values
        all_display['Change %'] = (all_display['Market Value'] - all_display['Prev Value']) / all_display['Prev Value'].abs() * 100
    else:
        prev_pos = pd.DataFrame()
        all_display['Prev Value'] = None
        all_display['Change %'] = None

    gross_portfolio = all_display['Market Value'].abs().sum()
    all_display['% of Portfolio'] = all_display['Market Value'] / gross_portfolio * 100

    nav_components = sec_total + cash_total + recv_total + liab_total
    prev_nav_total = all_display['Prev Value'].sum() if prev_nav_date else None
    nav_change_pct = (nav_components - prev_nav_total) / abs(prev_nav_total) * 100 if prev_nav_total else None

    total_row = pd.DataFrame([{
        'Account ID': '', 'Account Name': 'Total net assets', 'Quantity': None,
        'Market Price': None, 'Market Value': nav_components,
        'Prev Value': prev_nav_total, 'Change %': nav_change_pct,
        '% of Portfolio': 100.0
    }])
    all_display = pd.concat([all_display, total_row], ignore_index=True)

    position_units = units_row.iloc[0]['Quantity'] if len(units_row) > 0 else None

    def bold_total(row):
        if row['Account Name'] == 'Total net assets':
            return ['font-weight: bold'] * len(row)
        return [''] * len(row)

    display(all_display[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value', '% of Portfolio', 'Change %']].style
        .format({'Quantity': '{:,.2f}', 'Market Price': '{:.4f}', 'Market Value': '{:,.2f}',
                 '% of Portfolio': '{:.2f}%', 'Change %': '{:+.2f}%'}, na_rep='')
        .set_caption(f'{code} position as of {NAV_DATE} (vs {prev_nav_date})')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(bold_total, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['sec_total'] = sec_total
    fd['cash_total'] = cash_total
    fd['recv_total'] = recv_total
    fd['liab_total'] = liab_total
    fd['nav_components'] = nav_components
    fd['prev_nav_date'] = prev_nav_date
    fd['prev_nav_total'] = prev_nav_total
    fd['prev_pos'] = prev_pos if prev_nav_date else pd.DataFrame()
    fd['position_units'] = position_units

Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0,"9,039,054.83",15.5290,"140,367,482.44",28.94%,+0.50%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"4,053,375.60",34.5210,"139,926,579.09",28.85%,+0.52%
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,"9,031,547.00",12.0080,"108,450,816.38",22.36%,-0.33%
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible,"3,848,906.59",13.7070,"52,756,962.63",10.88%,-0.46%
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,"3,585,405.00",10.0900,"36,176,736.45",7.46%,-2.04%
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,"520,422.00",7.4510,"3,877,664.32",0.80%,-4.82%
,Cash,,,"2,189,971.98",0.45%,+16.50%
,Receivables,,,"795,752.34",0.16%,+inf%
,Liabilities,,,"-407,979.06",-0.08%,-inf%
,Total net assets,,,"484,133,986.57",100.00%,+0.12%


Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE00BJZ2DC62,Xtrackers MSCI USA Screened UCITS ETF,"23,687.00",50.1800,"1,188,613.66",16.51%,+11.35%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"34,096.56",34.5210,"1,177,047.35",16.35%,+0.52%
IE000F60HVH9,ICAV Amundi MSCI USA Screened UCITS ETF,"260,759.00",4.3405,"1,131,824.44",15.72%,-0.21%
IE000O58J820,Vanguard ESG North America All Cap UCITS ETF,"153,949.00",6.8540,"1,055,166.45",14.65%,+11.20%
LU1291099718,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,"45,082.00",18.8800,"851,148.16",11.82%,-3.07%
IE00BMDBMY19,Invesco MSCI Emerging Markets Universal Screened UCITS ETF Acc,"15,543.00",41.8300,"650,163.69",9.03%,-5.22%
LU1291102447,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,"11,704.00",18.0580,"211,350.83",2.93%,-4.62%
LU1291106356,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,"6,719.00",16.4560,"110,567.86",1.54%,-1.94%
LU0476289540,Xtrackers MSCI Canada Screened UCITS ETF,908.00,105.1400,"95,467.12",1.33%,-1.17%
,Cash,,,"494,157.25",6.86%,+13.73%


## Price Verification

Cross-checking fund position prices against independent index/provider data. These funds use NAV-date prices (no lag).

In [4]:
# ── Key mapping: ISIN → index keys per source ──
KEY_MAP = {
    # Shared: pooled BlackRock/iShares funds (TUV100 + TKF100)
    'IE0009FT4LX4': {
        'BlackRock': 'IE0009FT4LX4.BLACKROCK', 'EODHD': 'IE0009FT4LX4.EUFUND',
        'Morningstar': 'IE0009FT4LX4.MORNINGSTAR',
    },
    'IE00BFG1TM61': {
        'BlackRock': 'IE00BFG1TM61.BLACKROCK', 'EODHD': 'IE00BFG1TM61.EUFUND',
        'Morningstar': 'IE00BFG1TM61.MORNINGSTAR',
    },
    'IE00BKPTWY98': {
        'BlackRock': 'IE00BKPTWY98.BLACKROCK', 'EODHD': 'IE00BKPTWY98.EUFUND',
        'Morningstar': 'IE00BKPTWY98.MORNINGSTAR',
    },
    # TUV100 exchange-traded ETFs
    'IE00BFNM3G45': {
        'Exchange': 'IE00BFNM3G45.XETR', 'EODHD': 'SGAS.XETRA', 'Yahoo': 'SGAS.DE',
    },
    'IE00BFNM3D14': {
        'Exchange': 'IE00BFNM3D14.XETR', 'EODHD': 'SLMC.XETRA', 'Yahoo': 'SLMC.DE',
    },
    'IE00BFNM3L97': {
        'Exchange': 'IE00BFNM3L97.XETR', 'EODHD': 'SGAJ.XETRA', 'Yahoo': 'SGAJ.DE',
    },
    # TKF100 exchange-traded ETFs
    'IE00BJZ2DC62': {
        'Exchange': 'IE00BJZ2DC62.XETR', 'EODHD': 'XRSM.XETRA', 'Yahoo': 'XRSM.DE',
    },
    'IE000O58J820': {
        'Exchange': 'IE000O58J820.XETR', 'EODHD': 'V3YA.XETRA', 'Yahoo': 'V3YA.DE',
    },
    'IE000F60HVH9': {
        'Exchange': 'IE000F60HVH9.XPAR', 'EODHD': 'USAS.PA.EODHD', 'Yahoo': 'USAS.PA',
    },
    'LU1291106356': {
        'Exchange': 'LU1291106356.XETR', 'EODHD': 'PAC.XETRA', 'Yahoo': 'PAC.DE',
    },
    'IE00BMDBMY19': {
        'Exchange': 'IE00BMDBMY19.XETR', 'EODHD': 'ESGM.XETRA', 'Yahoo': 'ESGM.DE',
    },
    'LU1291102447': {
        'Exchange': 'LU1291102447.XETR', 'EODHD': 'EJAP.XETRA', 'Yahoo': 'EJAP.DE',
    },
    'LU1291099718': {
        'Exchange': 'LU1291099718.XETR', 'EODHD': 'EEUX.XETRA', 'Yahoo': 'EEUX.DE',
    },
    'LU0476289540': {
        'Exchange': 'LU0476289540.XETR', 'EODHD': 'D5BH.XETRA', 'Yahoo': 'D5BH.DE',
    },
}

# Positions where fund price is known stale — always reprice from providers,
# even from a single source (no consensus required)
ALWAYS_REPRICE = {'IE00BFG1TM61', 'IE0009FT4LX4', 'IE00BKPTWY98'}

SOURCES = ['BlackRock', 'Exchange', 'EODHD', 'Morningstar', 'Yahoo']

# Parse index data
idx = raw_index.copy()
idx['Date'] = pd.to_datetime(idx['Date'])
nav_dt = pd.to_datetime(NAV_DATE)

def latest_price_on_date(key, target_date):
    if key is None:
        return None
    rows = idx[(idx['Key'] == key) & (idx['Date'] == target_date)]
    if len(rows) == 0:
        return None
    return rows.iloc[0]['Value']

def find_consensus_price(alt_prices, threshold_pct=0.5):
    """Find consensus among ≥2 alternative prices that agree within threshold."""
    prices = [p for p in alt_prices if pd.notna(p)]
    if len(prices) < 2:
        return None
    for i in range(len(prices)):
        agreeing = [prices[i]]
        for j in range(len(prices)):
            if i != j and abs(prices[j] - prices[i]) / prices[i] * 100 <= threshold_pct:
                agreeing.append(prices[j])
        if len(agreeing) >= 2:
            return np.mean(agreeing)
    return None

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']

    verify_rows = []
    for _, sec in securities.sort_values('Market Value', ascending=False).iterrows():
        isin = sec['Account ID']
        mapping = KEY_MAP.get(isin, {})
        fund_price = sec['Market Price']

        # No lag — prices should be NAV date
        name = sec['Account Name']
        if isin in ALWAYS_REPRICE:
            name += ' *'
        row = {
            'ISIN': isin, 'Name': name, 'Fund': fund_price,
        }
        for src in SOURCES:
            row[src] = latest_price_on_date(mapping.get(src), nav_dt)
        verify_rows.append(row)

    verify_df = pd.DataFrame(verify_rows)

    def highlight_diff(row):
        fund_val = row['Fund']
        styles = [''] * len(row)
        for i, col in enumerate(row.index):
            if col in SOURCES and pd.notna(row[col]):
                diff_pct = abs(row[col] - fund_val) / fund_val * 100
                if diff_pct > 0.5:
                    styles[i] = 'background-color: #f8d7da'
                elif diff_pct > 0.01:
                    styles[i] = 'background-color: #fff3cd'
                else:
                    styles[i] = 'background-color: #d4edda'
        return styles

    if len(verify_df) > 0:
        display(verify_df.style
            .format({col: '{:.4f}' for col in ['Fund'] + SOURCES}, na_rep='—')
            .set_caption(f'{code} price verification (NAV date: {NAV_DATE})')
            .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
            .apply(highlight_diff, axis=1)
            .hide(axis='index'))

    n_ok = n_flag = n_nodata = 0
    for _, row in verify_df.iterrows():
        for src in SOURCES:
            val = row[src]
            if pd.isna(val) or val is None:
                n_nodata += 1
            elif abs(val - row['Fund']) / row['Fund'] * 100 > 0.5:
                n_flag += 1
            else:
                n_ok += 1

    # Compute repricing adjustment
    reprice_adj = 0.0
    for _, vrow in verify_df.iterrows():
        isin = vrow['ISIN']
        fund_price = vrow['Fund']
        alt_prices = [vrow[src] for src in SOURCES if src in vrow and pd.notna(vrow[src])]

        if isin in ALWAYS_REPRICE:
            # Fund price is known stale — use any available provider price
            nav_date_prices = [p for p in alt_prices if pd.notna(p)]
            if nav_date_prices:
                best_price = np.mean(nav_date_prices)
                qty_row = securities[securities['Account ID'] == isin]
                if len(qty_row) > 0:
                    reprice_adj += (best_price - fund_price) * qty_row.iloc[0]['Quantity']
        else:
            # Standard consensus repricing (≥2 sources agree, >0.5% diff)
            consensus = find_consensus_price(alt_prices)
            if consensus is not None and abs(consensus - fund_price) / fund_price * 100 > 0.5:
                qty_row = securities[securities['Account ID'] == isin]
                if len(qty_row) > 0:
                    reprice_adj += (consensus - fund_price) * qty_row.iloc[0]['Quantity']

    fd['verify_df'] = verify_df
    fd['n_ok'] = n_ok
    fd['n_flag'] = n_flag
    fd['n_nodata'] = n_nodata
    fd['reprice_adj'] = reprice_adj
    summary = f'{code}: {n_ok} OK  |  {n_flag} flagged  |  {n_nodata} no data'
    if abs(reprice_adj) > 0.01:
        summary += f'  |  reprice adj: {reprice_adj:+,.2f} EUR'
    print(summary)
    if any(vrow['ISIN'] in ALWAYS_REPRICE for _, vrow in verify_df.iterrows()):
        print(f'  (* = fund price is stale, always repriced from providers)')

ISIN,Name,Fund,BlackRock,Exchange,EODHD,Morningstar,Yahoo
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0 *,15.5290,—,—,—,15.4150,—
IE00BFG1TM61,iShares Developed World Screened Index Fund *,34.5210,—,—,—,34.2700,—
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,12.0080,—,12.0080,12.0080,—,12.0080
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible *,13.7070,—,—,—,13.3080,—
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,10.0900,—,10.0900,10.0900,—,10.0900
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,7.4510,—,7.4510,7.4510,—,7.4510


TUV100: 9 OK  |  3 flagged  |  18 no data  |  reprice adj: -3,583,563.26 EUR
  (* = fund price is stale, always repriced from providers)


ISIN,Name,Fund,BlackRock,Exchange,EODHD,Morningstar,Yahoo
IE00BJZ2DC62,Xtrackers MSCI USA Screened UCITS ETF,50.1800,—,50.1800,50.1800,—,50.1800
IE00BFG1TM61,iShares Developed World Screened Index Fund *,34.5210,—,—,—,34.2700,—
IE000F60HVH9,ICAV Amundi MSCI USA Screened UCITS ETF,4.3405,—,4.3405,4.3405,—,4.3405
IE000O58J820,Vanguard ESG North America All Cap UCITS ETF,6.8540,—,6.8540,6.8540,—,6.8540
LU1291099718,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,18.8800,—,18.8800,18.8800,—,18.8800
IE00BMDBMY19,Invesco MSCI Emerging Markets Universal Screened UCITS ETF Acc,41.8300,—,41.8300,41.8300,—,41.8300
LU1291102447,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,18.0580,—,18.0580,18.0580,—,18.0580
LU1291106356,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,16.4560,—,16.4560,16.4560,—,16.4560
LU0476289540,Xtrackers MSCI Canada Screened UCITS ETF,105.1400,—,105.1400,105.1400,—,105.1400


TKF100: 24 OK  |  1 flagged  |  20 no data  |  reprice adj: -8,558.24 EUR
  (* = fund price is stale, always repriced from providers)


## Net Asset Value Calculation

Computing NAV per unit from position data and comparing to the reported value. Fee accrual includes both management fee and depository bank fee (0.025% p.a.), accruing from the start of each month.

In [5]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    nav_components = fd['nav_components']
    position_units = fd['position_units']
    mgmt_fee_rate = fd['mgmt_fee_rate']
    prev_nav_date = fd['prev_nav_date']
    prev_nav_total = fd['prev_nav_total']
    prev_pos = fd['prev_pos']
    reprice_adj = fd.get('reprice_adj', 0)
    has_repricing = abs(reprice_adj) > 0.01
    adjustment = ADJUSTMENTS.get(code, 0)
    reported_nav = fd.get('reported_nav')
    nav_source = fd.get('nav_source', 'unavailable')

    nav_dt = datetime.strptime(NAV_DATE, '%Y-%m-%d')
    day_of_month = nav_dt.day
    total_fee_rate = mgmt_fee_rate + DEPOSITORY_FEE_RATE
    mgmt_accrual = nav_components * mgmt_fee_rate / 365 * day_of_month
    depo_accrual = nav_components * DEPOSITORY_FEE_RATE / 365 * day_of_month
    total_accrual = mgmt_accrual + depo_accrual
    nav_after_fees = nav_components - total_accrual + adjustment
    computed_nav = nav_after_fees / position_units

    nav_diff_eur = (computed_nav - reported_nav) if reported_nav else None
    nav_diff_pct = (nav_diff_eur / reported_nav * 100) if reported_nav else None

    # Repriced computation
    if has_repricing:
        repriced_components = nav_components + reprice_adj
        repriced_mgmt = repriced_components * mgmt_fee_rate / 365 * day_of_month
        repriced_depo = repriced_components * DEPOSITORY_FEE_RATE / 365 * day_of_month
        repriced_after_fees = repriced_components - repriced_mgmt - repriced_depo + adjustment
        repriced_nav = repriced_after_fees / position_units
        repriced_diff_eur = (repriced_nav - reported_nav) if reported_nav else None
        repriced_diff_pct = (repriced_diff_eur / reported_nav * 100) if reported_nav else None

    # Previous day
    prev_position_units = None
    prev_total_accrual = None
    prev_nav_after_fees = None
    prev_computed_nav = None
    if prev_nav_date and prev_nav_total:
        prev_units_rows = prev_pos[prev_pos['Account Type'] == 'UNITS'] if len(prev_pos) > 0 else pd.DataFrame()
        prev_position_units = prev_units_rows.iloc[0]['Quantity'] if len(prev_units_rows) > 0 else None
        prev_dt = datetime.strptime(prev_nav_date, '%Y-%m-%d')
        prev_total_accrual = prev_nav_total * total_fee_rate / 365 * prev_dt.day
        prev_nav_after_fees = prev_nav_total - prev_total_accrual
        if prev_position_units:
            prev_computed_nav = prev_nav_after_fees / prev_position_units

    def fmt_date(d):
        return datetime.strptime(d, '%Y-%m-%d').strftime('%d.%m.%Y') if d else 'Previous'

    def pct_change(cur, prev):
        if cur is not None and prev is not None and prev != 0:
            return f'{(cur - prev) / abs(prev) * 100:+.2f}%'
        return ''

    col_today = fmt_date(NAV_DATE)
    col_prev = fmt_date(prev_nav_date)

    reported_label = 'Total net assets as reported (EUR)' if has_repricing else 'Total net assets (EUR)'
    table_rows = [
        {'': reported_label, col_today: f'{nav_components:,.2f}', col_prev: f'{prev_nav_total:,.2f}' if prev_nav_total else '', 'Change': pct_change(nav_components, prev_nav_total)},
    ]
    if has_repricing:
        table_rows.append(
            {'': 'Total net assets repriced (EUR)', col_today: f'{repriced_components:,.2f}', col_prev: '', 'Change': f'{reprice_adj:+,.2f}'},
        )
    table_rows += [
        {'': f'Accrued mgmt fee est. ({mgmt_fee_rate*100:.3f}% p.a.)', col_today: f'{mgmt_accrual:,.2f}', col_prev: '', 'Change': ''},
        {'': f'Accrued depository fee est. ({DEPOSITORY_FEE_RATE*100:.3f}% p.a.)', col_today: f'{depo_accrual:,.2f}', col_prev: '', 'Change': ''},
        {'': 'Total accrued fees (EUR)', col_today: f'{total_accrual:,.2f}', col_prev: f'{prev_total_accrual:,.2f}' if prev_total_accrual else '', 'Change': pct_change(total_accrual, prev_total_accrual)},
    ]
    if adjustment:
        table_rows.append(
            {'': 'Adjustment (outstanding invoices)', col_today: f'{adjustment:+,.2f}', col_prev: '', 'Change': ''},
        )
    table_rows += [
        {'': 'Net assets after fees (EUR)', col_today: f'{nav_after_fees:,.2f}', col_prev: f'{prev_nav_after_fees:,.2f}' if prev_nav_after_fees else '', 'Change': pct_change(nav_after_fees, prev_nav_after_fees)},
        {'': 'Units outstanding', col_today: f'{position_units:,.2f}', col_prev: f'{prev_position_units:,.2f}' if prev_position_units else '', 'Change': pct_change(position_units, prev_position_units)},
        {'': 'Computed NAV (EUR)', col_today: f'{computed_nav:.5f}', col_prev: f'{prev_computed_nav:.5f}' if prev_computed_nav else '', 'Change': pct_change(computed_nav, prev_computed_nav)},
    ]
    if reported_nav:
        nav_source_label = f' (from {nav_source})' if nav_source != 'units card' else ''
        table_rows += [
            {'': f'Reported NAV (EUR){nav_source_label}', col_today: f'{reported_nav:.5f}', col_prev: '', 'Change': ''},
            {'': 'Difference (EUR)', col_today: f'{nav_diff_eur:+.5f}', col_prev: '', 'Change': f'{nav_diff_pct:+.3f}%'},
        ]
    else:
        table_rows.append({'': 'Reported NAV', col_today: 'not available', col_prev: '', 'Change': ''})
    if has_repricing and reported_nav:
        table_rows += [
            {'': 'Computed NAV repriced (EUR)', col_today: f'{repriced_nav:.5f}', col_prev: '', 'Change': ''},
            {'': 'Repriced difference (EUR)', col_today: f'{repriced_diff_eur:+.5f}', col_prev: '', 'Change': f'{repriced_diff_pct:+.3f}%'},
        ]

    table_df = pd.DataFrame(table_rows)

    ct = col_today
    def style_table(row, col_t=ct):
        styles = [''] * len(row)
        for i, col_name in enumerate(row.index):
            if col_name == col_t:
                styles[i] = 'font-weight: bold'
            elif col_name == 'Change':
                styles[i] = 'font-style: italic'
        return styles

    display(table_df.style
        .set_caption(f'{code} NAV calculation')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(style_table, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['computed_nav'] = computed_nav
    fd['nav_diff_eur'] = nav_diff_eur
    fd['nav_diff_pct'] = nav_diff_pct
    fd['total_accrual'] = total_accrual
    fd['total_units'] = position_units

,03.03.2026,02.03.2026,Change
Total net assets as reported (EUR),"484,133,986.57","483,564,191.82",+0.12%
Total net assets repriced (EUR),"480,550,423.31",,"-3,583,563.26"
Accrued mgmt fee est. (0.179% p.a.),"7,122.74",,
Accrued depository fee est. (0.025% p.a.),994.80,,
Total accrued fees (EUR),"8,117.53","5,405.32",+50.18%
Adjustment (outstanding invoices),"-9,676.30",,
Net assets after fees (EUR),"484,116,192.74","483,558,786.50",+0.12%
Units outstanding,"379,580,223.65","378,960,901.08",+0.16%
Computed NAV (EUR),1.27540,1.27601,-0.05%
Reported NAV (EUR),1.27870,,


,03.03.2026,02.03.2026,Change
Total net assets as reported (EUR),"6,729,815.37","6,751,012.55",-0.31%
Total net assets repriced (EUR),"6,721,257.13",,"-8,558.24"
Accrued mgmt fee est. (0.150% p.a.),82.97,,
Accrued depository fee est. (0.025% p.a.),13.83,,
Total accrued fees (EUR),96.80,64.74,+49.53%
Net assets after fees (EUR),"6,729,718.57","6,750,947.81",-0.31%
Units outstanding,"6,800,722.76","6,741,179.24",+0.88%
Computed NAV (EUR),0.98956,1.00145,-1.19%
Reported NAV (EUR) (from index data (2026-03-03)),0.98830,,
Difference (EUR),+0.00126,,+0.127%


## Day-over-Day Comparison

Comparing fund NAV change to MSCI ACWI previous-day change. No lag adjustment needed — both funds use NAV-date prices.

In [6]:
idx_data = raw_index.copy()
idx_data['Date'] = pd.to_datetime(idx_data['Date'])

def clean_series(series):
    """Remove outlier rows where price jumps >20% from neighbours."""
    s = series.sort_values('Date').reset_index(drop=True)
    if len(s) < 3:
        return s
    mask = pd.Series(True, index=s.index)
    for i in range(1, len(s) - 1):
        prev_v, cur_v, next_v = s.loc[i-1, 'Value'], s.loc[i, 'Value'], s.loc[i+1, 'Value']
        if prev_v > 0 and next_v > 0:
            chg_prev = abs(cur_v - prev_v) / prev_v
            chg_next = abs(cur_v - next_v) / next_v
            if chg_prev > 0.20 and chg_next > 0.20:
                mask[i] = False
    if len(s) >= 2:
        last_v, prev_v = s.loc[len(s)-1, 'Value'], s.loc[len(s)-2, 'Value']
        if prev_v > 0 and abs(last_v - prev_v) / prev_v > 0.20:
            mask[len(s)-1] = False
    dropped = (~mask).sum()
    if dropped > 0:
        print(f'  \u26a0 Dropped {dropped} outlier(s) from price series')
    return s[mask]

# Find MSCI ACWI index
acwi_key = 'MSCI_ACWI'
acwi_series = clean_series(idx_data[idx_data['Key'] == acwi_key])
nav_dt_ts = pd.to_datetime(NAV_DATE)

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    prev_date = fd['prev_date']
    reported_nav = fd.get('reported_nav')

    print(f'\u2550\u2550\u2550 {code} \u2550\u2550\u2550')

    if reported_nav is None:
        print(f'Cannot compare: no reported NAV available\n')
        fd['fund_nav_change_pct'] = None
        continue

    # Get previous NAV — from units card or index data
    prev_nav = None
    if len(fd['units_prev']) > 0:
        prev_nav = fd['units_prev'].iloc[0]['Nav']
    elif 'prev_nav_from_index' in fd:
        prev_nav = fd['prev_nav_from_index']

    if prev_nav is None or prev_date is None:
        print(f'Cannot compare: no previous NAV available\n')
        fd['fund_nav_change_pct'] = None
        continue

    today_nav = reported_nav
    fund_nav_change_pct = (today_nav - prev_nav) / prev_nav * 100

    print(f'Fund NAV: {prev_date} {prev_nav:.6f} \u2192 {NAV_DATE} {today_nav:.6f}  ({fund_nav_change_pct:+.2f}%)')

    prev_dt_ts = pd.to_datetime(prev_date)

    # Simple MSCI ACWI comparison — no lag blending
    acwi_on_nav = acwi_series[acwi_series['Date'] <= nav_dt_ts]
    acwi_on_prev = acwi_series[acwi_series['Date'] <= prev_dt_ts]

    if len(acwi_on_nav) > 0 and len(acwi_on_prev) > 0:
        acwi_today_row = acwi_on_nav.iloc[-1]
        acwi_prev_row = acwi_on_prev.iloc[-1]

        if acwi_today_row['Date'] != acwi_prev_row['Date']:
            acwi_change_pct = (acwi_today_row['Value'] - acwi_prev_row['Value']) / acwi_prev_row['Value'] * 100
        else:
            acwi_change_pct = 0.0

        tracking_diff = fund_nav_change_pct - acwi_change_pct

        rows = [
            {'Metric': 'Fund NAV change', 'Value': f'{fund_nav_change_pct:+.2f}%'},
            {'Metric': f'MSCI ACWI ({acwi_prev_row["Date"].strftime("%m-%d")}\u2192{acwi_today_row["Date"].strftime("%m-%d")})', 'Value': f'{acwi_change_pct:+.2f}%'},
            {'Metric': 'Tracking difference', 'Value': f'{tracking_diff:+.2f}%'},
        ]

        comparison = pd.DataFrame(rows)
        display(comparison.style.hide(axis='index'))

        fd['acwi_change_pct'] = acwi_change_pct
        fd['tracking_diff'] = tracking_diff
    else:
        print('MSCI ACWI data insufficient for comparison')

    fd['fund_nav_change_pct'] = fund_nav_change_pct
    print()

═══ TUV100 ═══
Fund NAV: 2026-03-02 1.276400 → 2026-03-03 1.278700  (+0.18%)


Metric,Value
Fund NAV change,+0.18%
MSCI ACWI (03-02→03-03),-1.00%
Tracking difference,+1.18%



═══ TKF100 ═══
Fund NAV: 2026-03-02 1.001500 → 2026-03-03 0.988300  (-1.32%)


Metric,Value
Fund NAV change,-1.32%
MSCI ACWI (03-02→03-03),-1.00%
Tracking difference,-0.31%
